# Notebook 06: Phase 2 data (E7) and pre-registration

**Protocol v3.** This notebook first writes the pre-registration to Google Drive with a timestamp and SHA-256 (cell 1), then:

1. downloads the registered KaraAgro subset (`maize_streak_01.rar`, `healthy_01.rar`, `faw_01.zip`, about 6.5 GB) and checks each archive's MD5 against the repository;
2. downloads the unused CCMT folders (cassava brown spot and green mite; maize fall armyworm, grasshopper, leaf beetle, leaf blight, leaf spot; cashew healthy and anthracnose; tomato healthy and leaf curl) with per-file SHA-256 checks;
3. inventories every image, makes working copies, finds exact and near duplicates against **all** Phase 1 images, quarantines copies, and groups the rest;
4. writes `manifest_v3.csv`: the Phase 1 manifest plus the new images, with the registered group-wise 50/50 `adapt`/`test` splits of D3 (per crop) and D5, the challenge set, and the **sealed** `d5_test` half, which no later notebook may open before v3 is frozen.

**Runtime:** CPU is enough. About 45 to 75 minutes, mostly downloading. Finished steps are skipped if you run all again.

In [ ]:
# 1. Pre-registration: written once, before anything else runs
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, time, glob, tarfile, hashlib, datetime, platform, subprocess, shutil, zipfile, csv, collections
import numpy as np, pandas as pd
DRIVE = '/content/drive/MyDrive/ocdf'; RAW = '/content/raw'; WORK = '/content/working'
OUT = f'{DRIVE}/records/nb06'
for d in [OUT, RAW, WORK]: os.makedirs(d, exist_ok=True)
REGISTRATION_TEXT = ''' + repr(REG_MD) + r'''
reg_path = f'{DRIVE}/records/protocol_v3_registration.json'
if not os.path.exists(reg_path):
    reg = {'protocol_version': 'v3', 'registered_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
           'builds_on': {'manifest_v2_sha256': json.load(open(f'{DRIVE}/records/protocol_v2.json'))['manifest_sha256'],
                         'frozen_choices_v2_sha256': open(f'{DRIVE}/records/frozen_choices_v2.sha256').read().strip()},
           'registered_d5_archives': ['maize_streak_01.rar', 'healthy_01.rar', 'faw_01.zip'],
           'hypotheses': ['H1 transfer matrix', 'H2 background counterfactual', 'H3 label-free adaptation', 'H4 selective INT8 + phone', 'H5 abstention under shift'],
           'primary_confirmatory_endpoint': 'H3: paired group-bootstrap macro-F1 (MSV + healthy) on d5_test, adapted vs unadapted Phase 1 model; '
                                            'confirmed if 95% CI excludes 0 and estimate >= +0.02',
           'split_seed': 2026, 'split_rule': 'd3c, d3m, d5: group-wise, stratified by label, 50/50 adapt/test',
           'registration_text_markdown': REGISTRATION_TEXT}
    json.dump(reg, open(reg_path, 'w'), indent=1)
    open(reg_path + '.sha256', 'w').write(hashlib.sha256(open(reg_path, 'rb').read()).hexdigest())
    print('REGISTERED at', reg['registered_utc'])
reg = json.load(open(reg_path)); reg_sha = open(reg_path + '.sha256').read().strip()
assert hashlib.sha256(open(reg_path, 'rb').read()).hexdigest() == reg_sha
print('registration', reg['registered_utc'], 'SHA-256', reg_sha[:16], '...')
env = {'notebook': '06_phase2_data', 'run_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'python': sys.version.split()[0],
       'platform': platform.platform(), 'cpu_count': os.cpu_count(), 'numpy': np.__version__, 'pandas': pd.__version__}
json.dump(env, open(f'{OUT}/environment_nb06.json', 'w'), indent=1); print(env)
def done(step): return os.path.exists(f'{OUT}/{step}.done')
def mark(step, info=None): json.dump(info or {}, open(f'{OUT}/{step}.done', 'w'), indent=1, default=str)

In [ ]:
# 2. Helpers (identical to Phase 1)
# ---- OCDF F2 inventory helpers (Notebook 01) ----
import hashlib, io, os, csv, json
import numpy as np
from PIL import Image, ImageOps

WORKING_MAX_SIDE = 512        # working copies: longest side <= 512 px, aspect kept
WORKING_JPEG_QUALITY = 95
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

_DCT32 = None
def _dct_matrix(n=32):
    # Unnormalised DCT-II, identical to scipy.fftpack.dct (as used by imagehash.phash)
    k = np.arange(n)[:, None]; i = np.arange(n)[None, :]
    return 2.0 * np.cos(np.pi * (2 * i + 1) * k / (2 * n))

def phash_hex(img):
    """64-bit DCT perceptual hash (same method as imagehash.phash, hash_size=8)."""
    global _DCT32
    if _DCT32 is None:
        _DCT32 = _dct_matrix(32)
    g = np.asarray(img.convert("L").resize((32, 32), Image.LANCZOS), dtype=np.float64)
    d = _DCT32 @ g @ _DCT32.T
    low = d[:8, :8]
    bits = (low > np.median(low)).flatten()
    return "%016x" % int("".join("1" if b else "0" for b in bits), 2)

def hamming(h1, h2):
    return bin(int(h1, 16) ^ int(h2, 16)).count("1")

def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()

def md5_file(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()

def inspect_image(raw_bytes, working_out=None):
    """Decode one original image; return inventory fields. Optionally write a working copy.

    The original bytes are never modified. The working copy is EXIF-orientation corrected,
    RGB, longest side <= WORKING_MAX_SIDE, saved as JPEG.
    """
    rec = {"sha256": sha256_bytes(raw_bytes), "bytes": len(raw_bytes), "decode_ok": False,
           "format": "", "width": "", "height": "", "mode": "", "phash": "", "decode_error": "",
           "working_path": "", "working_width": "", "working_height": ""}
    try:
        with Image.open(io.BytesIO(raw_bytes)) as im:
            rec["format"] = im.format or ""
            rec["width"], rec["height"] = im.size
            rec["mode"] = im.mode
            im.load()                                   # full decode = integrity check
            im = ImageOps.exif_transpose(im)
            rgb = im.convert("RGB")
        rec["phash"] = phash_hex(rgb)
        rec["decode_ok"] = True
        if working_out:
            w = rgb.copy()
            w.thumbnail((WORKING_MAX_SIDE, WORKING_MAX_SIDE), Image.LANCZOS)
            os.makedirs(os.path.dirname(working_out), exist_ok=True)
            w.save(working_out, "JPEG", quality=WORKING_JPEG_QUALITY)
            rec["working_path"] = working_out
            rec["working_width"], rec["working_height"] = w.size
    except Exception as e:                              # corrupt / unreadable file
        rec["decode_error"] = f"{type(e).__name__}: {e}"[:200]
    return rec

INVENTORY_FIELDS = ["source_id", "release", "original_relpath", "original_label", "research_label",
                    "crop", "selected", "exclusion_reason", "sha256", "bytes", "decode_ok", "decode_error",
                    "format", "width", "height", "mode", "phash", "working_path", "working_width",
                    "working_height", "group_hint", "published_checksum", "checksum_match"]

def write_inventory(rows, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=INVENTORY_FIELDS, extrasaction="ignore")
        w.writeheader()
        for r in rows:
            w.writerow(r)

def count_table(rows):
    """Per source x original label: downloaded, readable, selected (research label), excluded."""
    from collections import defaultdict
    t = defaultdict(lambda: {"downloaded": 0, "readable": 0, "unreadable": 0, "selected": 0, "excluded": 0})
    for r in rows:
        k = (r["source_id"], r["original_label"], r["research_label"] or "(excluded)")
        t[k]["downloaded"] += 1
        ok = str(r["decode_ok"]) == "True"
        t[k]["readable" if ok else "unreadable"] += 1
        sel = str(r["selected"]) == "True"          # CSV round-trip turns booleans into text
        t[k]["selected" if (sel and ok) else "excluded"] += 1
    return [dict(source_id=k[0], original_label=k[1], research_label=k[2], **v) for k, v in sorted(t.items())]

# ---- OCDF F2 duplicate grouping and partitioning helpers (Notebook 02) ----
import random, collections
import numpy as np

DEV_SOURCES = ('D1', 'D4', 'D6')
EXT_SOURCES = ('D3',)
PARTITIONS = [('train', 0.70), ('select', 0.10), ('calib_T', 0.05), ('thresh', 0.05), ('test', 0.10)]
SOURCE_PRIORITY = {'D1': 0, 'D4': 1, 'D6': 2, 'D3': 3}

class UnionFind:
    def __init__(self, n): self.p = list(range(n))
    def find(self, a):
        while self.p[a] != a:
            self.p[a] = self.p[self.p[a]]; a = self.p[a]
        return a
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb: self.p[max(ra, rb)] = min(ra, rb)

def near_duplicate_pairs(hashes, max_dist=8, chunk=1500):
    """All index pairs (a < b) with Hamming distance <= max_dist between 64-bit hex hashes.

    Exhaustive comparison in chunks with NumPy's bitwise_count (NumPy 2.0 or later).
    """
    H = np.array([int(h, 16) for h in hashes], dtype=np.uint64)
    pairs = []
    for i0 in range(0, len(H), chunk):
        blk = H[i0:i0 + chunk]
        d = np.bitwise_count(blk[:, None] ^ H[None, :])
        ii, jj = np.nonzero(d <= max_dist)
        for a, b in zip(ii + i0, jj):
            if a < b: pairs.append((int(a), int(b), int(d[a - i0, b])))
    return pairs

def grouped_split(groups_by_stratum, seed=2026, partitions=PARTITIONS):
    """Assign whole groups to partitions, per stratum, approximating the target shares by image count.

    groups_by_stratum: {stratum: [(group_id, n_images), ...]}. Returns {group_id: partition}.
    Groups are shuffled with the seed, then placed largest first into the partition whose share is
    furthest below its target. Group integrity always wins over exact shares.
    """
    rng = random.Random(seed)
    out = {}
    names = [p for p, _ in partitions]; target = dict(partitions)
    for stratum in sorted(groups_by_stratum):
        gs = list(groups_by_stratum[stratum]); rng.shuffle(gs)
        gs.sort(key=lambda g: -g[1])                     # stable: equal sizes keep shuffled order
        total = sum(n for _, n in gs); filled = {p: 0 for p in names}
        for gid, n in gs:
            # deficit after placing: choose partition with the largest remaining shortfall
            best = max(names, key=lambda p: (target[p] * total - filled[p]) / max(target[p], 1e-9))
            out[gid] = best; filled[best] += n
    return out

def class_weights(train_counts, cap=5.0):
    """Training recipe: inverse-frequency weights normalised as n / (k * n_c) (so the image-weighted mean is 1),
    then capped at `cap`."""
    labels = sorted(train_counts); n = sum(train_counts.values()); k = len(labels)
    return {c: round(min(n / (k * train_counts[c]), cap), 4) for c in labels}

In [ ]:
# 3. Phase 1 manifest (for duplicate checks) and the new label map
protocol2 = json.load(open(f'{DRIVE}/records/protocol_v2.json'))
M2 = pd.read_csv(f'{DRIVE}/manifests/{protocol2["manifest"]}', dtype=str)
assert hashlib.sha256(open(f'{DRIVE}/manifests/{protocol2["manifest"]}', 'rb').read()).hexdigest() == protocol2['manifest_sha256']
print('Phase 1 manifest rows:', len(M2))
# role: challenge_near_cassava / challenge_near_maize / challenge_far / d5_sealed. research_label for D5 maps to the maize task.
NEW_LABELS = {
  ('D3x', 'Cassava/brown spot'):   ('', 'cassava', 'challenge_near_cassava'),
  ('D3x', 'Cassava/green mite'):   ('', 'cassava', 'challenge_near_cassava'),
  ('D3x', 'Maize/fall armyworm'):  ('', 'maize', 'challenge_near_maize'),
  ('D3x', 'Maize/grasshoper'):     ('', 'maize', 'challenge_near_maize'),
  ('D3x', 'Maize/leaf beetle'):    ('', 'maize', 'challenge_near_maize'),
  ('D3x', 'Maize/leaf blight'):    ('', 'maize', 'challenge_near_maize'),
  ('D3x', 'Maize/leaf spot'):      ('', 'maize', 'challenge_near_maize'),
  ('D3x', 'Cashew/healthy'):       ('', 'other', 'challenge_far'),
  ('D3x', 'Cashew/anthracnose'):   ('', 'other', 'challenge_far'),
  ('D3x', 'Tomato/healthy'):       ('', 'other', 'challenge_far'),
  ('D3x', 'Tomato/leaf curl'):     ('', 'other', 'challenge_far'),
  ('D5', 'maize_streak'):          ('M01', 'maize', 'd5'),
  ('D5', 'healthy'):               ('M05', 'maize', 'd5'),
  ('D5', 'faw'):                   ('', 'maize', 'challenge_near_maize'),
}
LABEL_MAP = {k: (v[0], v[1]) for k, v in NEW_LABELS.items()}
EXCLUSION_REASON = {}

In [ ]:
# 4. Inventory function (working copies are made for every image here, since all are used for evaluation)
from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mp
def _work(args):
    src, release, relpath, olabel, fpath, extra = args
    rlabel, crop = LABEL_MAP.get((src, olabel), ('', ''))
    with open(fpath, 'rb') as f: raw = f.read()
    sha = sha256_bytes(raw)
    wout = f'{WORK}/{src}/{olabel.replace("/", "_").replace(" ", "_")}/{sha[:20]}.jpg'
    rec = inspect_image(raw, wout)
    rec.update(source_id=src, release=release, original_relpath=relpath, original_label=olabel, research_label=rlabel, crop=crop,
               selected=True, exclusion_reason='', group_hint=extra.get('group_hint', ''), published_checksum=extra.get('published', ''),
               checksum_match=('' if not extra.get('published') else str(extra['published'] == sha)))
    if rec['working_path']: rec['working_path'] = os.path.relpath(rec['working_path'], WORK)
    return rec
def inventory(src, release, items):
    t0 = time.time()
    with ProcessPoolExecutor(max_workers=os.cpu_count(), mp_context=mp.get_context('fork')) as ex:
        rows = list(ex.map(_work, [(src, release, *it) for it in items], chunksize=16))
    print(f'{src}: {len(rows)} images ({sum(r["decode_ok"] for r in rows)} readable) in {time.time()-t0:.0f}s'); return rows
def pack_working(src):
    tp = f'{DRIVE}/working/working_{src}.tar'
    with tarfile.open(tp, 'w') as tar: tar.add(f'{WORK}/{src}', arcname=src)
    return {'tar': os.path.basename(tp), 'sha256': sha256_file(tp), 'bytes': os.path.getsize(tp)}
import requests
def download(url, out, tries=5):
    for t in range(tries):
        try:
            with requests.get(url, stream=True, timeout=180) as r:
                r.raise_for_status()
                with open(out + '.part', 'wb') as f:
                    for blk in r.iter_content(1 << 20): f.write(blk)
            os.replace(out + '.part', out); return
        except Exception as e:
            print(f'  retry {t+1} for {os.path.basename(out)}: {e}'); time.sleep(5 * (t + 1))
    raise RuntimeError(f'Download failed: {url}')

In [ ]:
# 5. CCMT unused folders (D3x): per-file SHA-256 from Mendeley
from concurrent.futures import ThreadPoolExecutor
D3X_FOLDERS = {
  'Cassava/brown spot': 'd871023f-5dc4-4e10-a4b9-c0b1762ce351', 'Cassava/green mite': '3d09905e-e4f5-4415-a454-120fd29b6951',
  'Maize/fall armyworm': '7b44bb20-452c-468d-9caf-9149ea7b21e7', 'Maize/grasshoper': '4da8265a-30d2-49bc-ace2-dbcec2d53aab',
  'Maize/leaf beetle': '4cf175af-5a5a-4191-a3ce-f202b75f0d85', 'Maize/leaf blight': '08044ee4-b294-4135-b6d4-b7e6ed689618',
  'Maize/leaf spot': '632f07ab-cafd-4752-81ed-8f9914ce64e6', 'Cashew/healthy': '174b1f48-76a5-42cd-8b39-0f1d2adc2da0',
  'Cashew/anthracnose': 'a1507961-25fd-4ccc-b4ea-dbbc9cbf1aa4', 'Tomato/healthy': 'aa41f1e8-8303-4e3a-8fb6-61828fd133a7',
  'Tomato/leaf curl': '03ac0d50-4313-47f2-9afb-4bf3166f7fa1'}
API = 'https://data.mendeley.com/public-api/datasets/bwh3zbpkpv'
if not done('inventory_D3x'):
    d3 = f'{RAW}/D3x'; listing = []
    for label, fid in D3X_FOLDERS.items():
        start = 0
        while True:
            r = requests.get(f'{API}/files', params={'folder_id': fid, 'version': 1, '$start': start, '$limit': 1000}, timeout=60)
            r.raise_for_status(); page = r.json(); start += len(page)
            listing += [(label, f['filename'], f['content_details']['download_url'], f['content_details']['sha256_hash']) for f in page]
            if len(page) < 1000: break
        print(f'{label}: {sum(1 for x in listing if x[0] == label)} files')
    def fetch(x):
        label, fname, url, sha_pub = x; out = f'{d3}/{label}/{fname}'
        if not (os.path.exists(out) and sha256_file(out) == sha_pub):
            os.makedirs(os.path.dirname(out), exist_ok=True); download(url, out)
        return out
    t0 = time.time()
    with ThreadPoolExecutor(16) as ex: paths = list(ex.map(fetch, listing))
    print(f'downloaded {len(paths)} files in {time.time()-t0:.0f}s')
    rows = inventory('D3x', 'Mendeley Data version 1 (unused folders)', [(f'Raw Data/CCMT Dataset/{lab}/{fn}', lab, p, {'published': sha}) for (lab, fn, u, sha), p in zip(listing, paths)])
    bad = sum(1 for r in rows if r['checksum_match'] != 'True'); print('SHA-256 mismatches:', bad)
    write_inventory(rows, f'{OUT}/inventory_D3x.csv'); mark('inventory_D3x', {'working': pack_working('D3x'), 'n': len(rows), 'mismatches': bad})
print('D3x:', json.load(open(f'{OUT}/inventory_D3x.done')))

In [ ]:
# 6. KaraAgro (D5) registered subset: three archives, MD5-checked, extracted, inventoried
D5_FILES = {'maize_streak_01.rar': (6417320, '628da79bf49b0f4caf1b1c54eeedadbe', 'maize_streak'),
            'healthy_01.rar':      (6417970, '96ffe973414f0daee34adfda505c09cc', 'healthy'),
            'faw_01.zip':          (6418125, '4915771858f68b3a43a11da6fffb2c3a', 'faw')}
assert list(D5_FILES) == reg['registered_d5_archives']
if not done('inventory_D5'):
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'unrar'], check=False, capture_output=True)
    d5 = f'{RAW}/D5'; os.makedirs(d5, exist_ok=True); arch = {}; items = []
    for name, (fid, md5_pub, label) in D5_FILES.items():
        ap = f'{d5}/{name}'
        if not os.path.exists(ap): print('downloading', name); download(f'https://dataverse.harvard.edu/api/access/datafile/{fid}', ap)
        md5 = md5_file(ap); assert md5 == md5_pub, f'{name}: MD5 {md5} differs from the published {md5_pub}; download again'
        arch[name] = {'dataverse_file_id': fid, 'md5': md5, 'published_md5': md5_pub, 'md5_match': True, 'sha256': sha256_file(ap),
                      'bytes': os.path.getsize(ap), 'retrieved_utc': datetime.datetime.now(datetime.timezone.utc).isoformat()}
        out = f'{d5}/x/{label}'; os.makedirs(out, exist_ok=True)
        if not glob.glob(f'{out}/**/*', recursive=True):
            if name.endswith('.zip'):
                with zipfile.ZipFile(ap) as z: z.extractall(out)
            else:
                r = subprocess.run(['unrar', 'x', '-o+', '-y', ap, out + '/'], capture_output=True, text=True)
                assert r.returncode == 0, 'unrar failed: ' + r.stderr[-500:]
        files = [p for p in glob.glob(f'{out}/**/*', recursive=True) if os.path.isfile(p) and os.path.splitext(p)[1].lower() in IMG_EXT and '__MACOSX' not in p]
        print(f'{name}: {len(files)} image files, MD5 ok')
        items += [(os.path.relpath(p, d5), label, p, {'group_hint': os.path.dirname(os.path.relpath(p, out))}) for p in sorted(files)]
    rows = inventory('D5', 'Harvard Dataverse version 4.1, registered subset', items)
    write_inventory(rows, f'{OUT}/inventory_D5.csv'); mark('inventory_D5', {'archives': arch, 'working': pack_working('D5'), 'n': len(rows)})
print('D5:', {k: v for k, v in json.load(open(f'{OUT}/inventory_D5.done')).items() if k != 'archives'})

In [ ]:
# 7. Duplicates against all Phase 1 images, quarantine, grouping, roles, and the manifest addition
N = pd.concat([pd.read_csv(f'{OUT}/inventory_{s}.csv', dtype=str, keep_default_na=False) for s in ['D3x', 'D5']], ignore_index=True)
N = N[N.decode_ok == 'True'].copy(); N['sample_id'] = [f'{s}-{i:06d}' for i, s in enumerate(N.source_id)]
N['role'] = [NEW_LABELS[(s, l)][2] for s, l in zip(N.source_id, N.original_label)]
excl = []
# exact duplicates: against Phase 1 and within the new images
old_sha = set(M2.sha256); dup_old = N.sha256.isin(old_sha)
for r in N[dup_old].itertuples(): excl.append({'sample_id': r.sample_id, 'source_id': r.source_id, 'original_label': r.original_label, 'reason': 'exact copy of a Phase 1 image'})
N = N[~dup_old]; first = ~N.duplicated('sha256', keep='first')
for r in N[~first].itertuples(): excl.append({'sample_id': r.sample_id, 'source_id': r.source_id, 'original_label': r.original_label, 'reason': 'exact duplicate within Phase 2 data'})
N = N[first].reset_index(drop=True)
# near duplicates: new vs old (quarantine at d<=6) and new vs new (group at d<=8, quarantine label conflicts at d<=6)
ALL = pd.concat([M2[['sample_id', 'source_id', 'research_label', 'phash']].assign(role='phase1', original_label=M2.research_label),
                 N[['sample_id', 'source_id', 'research_label', 'phash', 'role', 'original_label']]], ignore_index=True)
pairs = near_duplicate_pairs(list(ALL.phash), 8)
is_new = (ALL.role != 'phase1').values; lab = ALL.original_label.values; role = ALL.role.values
quar = {}; uf = UnionFind(len(ALL)); stats = {'new_vs_phase1_d<=6': 0, 'new_vs_phase1_d8': 0, 'new_vs_new_d<=8': 0, 'new_vs_new_label_conflict_d<=6': 0}
for a, b, d in pairs:
    if is_new[a] and is_new[b]:
        stats['new_vs_new_d<=8'] += 1
        if lab[a] != lab[b] and d <= 6:
            stats['new_vs_new_label_conflict_d<=6'] += 1
            quar.setdefault(ALL.sample_id[a], f'near duplicate (d={d}) of an image with another label'); quar.setdefault(ALL.sample_id[b], f'near duplicate (d={d}) of an image with another label')
        elif lab[a] == lab[b]: uf.union(a, b)
    elif is_new[a] or is_new[b]:
        new_i = a if is_new[a] else b
        if d <= 6: stats['new_vs_phase1_d<=6'] += 1; quar.setdefault(ALL.sample_id[new_i], f'near duplicate (d={d}) of a Phase 1 image')
        else: stats['new_vs_phase1_d8'] += 1
for r in N[N.sample_id.isin(quar)].itertuples(): excl.append({'sample_id': r.sample_id, 'source_id': r.source_id, 'original_label': r.original_label, 'reason': quar[r.sample_id]})
N = N[~N.sample_id.isin(quar)].copy()
root = {ALL.sample_id[i]: uf.find(i) for i in range(len(ALL)) if is_new[i]}
roots = {rt: f'H{k:06d}' for k, rt in enumerate(sorted(set(root[s] for s in N.sample_id)))}
N['group_id'] = [roots[root[s]] for s in N.sample_id]
# Registered splits (seed 2026, group-wise, stratified by label): D3 cassava, D3 maize (from the Phase 1 manifest) and D5 -> adapt / test halves
def half_split(df, seed=2026):
    strata = collections.defaultdict(list)
    for gid, g in df.groupby('group_id'): strata[(g.source_id.iloc[0], g.research_label.iloc[0])].append((gid, len(g)))
    return grouped_split(strata, seed=seed, partitions=[('adapt', 0.5), ('test', 0.5)])
M3 = M2.copy()
d3c = M3[(M3.source_id == 'D3') & (M3.crop == 'cassava')]; d3m = M3[(M3.source_id == 'D3') & (M3.crop == 'maize')]
a = half_split(d3c); M3.loc[d3c.index, 'partition'] = ['d3c_' + a[g] for g in d3c.group_id]
a = half_split(d3m); M3.loc[d3m.index, 'partition'] = ['d3m_' + a[g] for g in d3m.group_id]
N['partition'] = N.role
d5 = N[N.role == 'd5']; a = half_split(d5); N.loc[d5.index, 'partition'] = ['d5_' + a[g] for g in d5.group_id]
cols = ['sample_id', 'source_id', 'crop', 'original_label', 'research_label', 'partition', 'group_id', 'working_path', 'sha256', 'phash', 'width', 'height', 'original_relpath']
X = N[cols].sort_values(['partition', 'sample_id'])
FULL = pd.concat([M3[cols], X], ignore_index=True)
xpath = f'{DRIVE}/manifests/manifest_v3.csv'; FULL.to_csv(xpath, index=False); X.to_csv(f'{DRIVE}/manifests/manifest_v3_extra.csv', index=False)
pd.DataFrame(excl).to_csv(f'{OUT}/exclusions_v3_extra.csv', index=False)
xsha = hashlib.sha256(open(xpath, 'rb').read()).hexdigest()
counts = FULL[FULL.partition.str.startswith(('d3c', 'd3m', 'd5', 'challenge'))].groupby(['partition', 'source_id', 'original_label']).agg(images=('sample_id', 'size'), groups=('group_id', 'nunique'))
print(counts); print(stats); print('excluded:', pd.DataFrame(excl).reason.value_counts().to_dict() if excl else 0)
seal = {'manifest_v3': os.path.basename(xpath), 'sha256': xsha, 'rows': int(len(FULL)), 'd5_test_images': int((FULL.partition == 'd5_test').sum()),
        'd5_test_groups': int(FULL[FULL.partition == 'd5_test'].group_id.nunique()), 'sealed_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
        'rule': 'd5_test rows may not be loaded, scored or inspected before frozen_choices_v3.json exists; d3c_test, d3m_test, d5_test are never trained on or adapted to',
        'near_duplicate_stats': stats, 'archives': {'D3x': json.load(open(f'{OUT}/inventory_D3x.done')), 'D5': json.load(open(f'{OUT}/inventory_D5.done'))},
        'registration_sha256': reg_sha, 'environment': env}
json.dump(seal, open(f'{DRIVE}/records/phase2_data_v3.json', 'w'), indent=1, default=str); counts.to_csv(f'{OUT}/phase2_counts.csv')
print('manifest_v3 frozen:', xsha)

In [ ]:
# 8. Results file (counts, exclusions, registration; no D5 image details beyond counts)
res = '/content/nb06_results.zip'
with zipfile.ZipFile(res, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in [reg_path, reg_path + '.sha256', f'{DRIVE}/records/phase2_data_v3.json', f'{OUT}/phase2_counts.csv', f'{OUT}/exclusions_v3_extra.csv',
              f'{OUT}/environment_nb06.json', xpath, f'{DRIVE}/manifests/manifest_v3_extra.csv']:
        if os.path.exists(p): z.write(p, os.path.basename(p))
from google.colab import files
files.download(res)

### When it finishes
The run records are written to Google Drive under `ocdf/records/` and also downloaded as `nb06_results.zip`. The copies used for the published results are in this repository under `records/nb06/`.